# Importing Libraries

In [140]:
import pandas as pd
radio = pd.read_csv("radio.csv")
youtube = pd.read_csv("youtube.csv")
audiomack = pd.read_csv("audiomack.csv")

# Data Cleaning 

In [141]:
radio = radio.rename(columns={
    "Title": "title",
    "Imp's": "radio_count",
})
youtube = youtube.rename(columns={
    "Song": "title",
    "Artiste": "artist",
    "Difference": "youtube_count"
})
audiomack = audiomack.rename(columns={
    "total": "audiomack_count"
})

In [142]:
radio["title"] = radio ["title"].str.lower().str.strip()
youtube["title"] = youtube["title"].str.lower().str.strip()
audiomack["title"] = audiomack["title"].str.lower().str.strip()

In [143]:
youtube["artist"] = youtube["artist"].str.lower().str.strip()
audiomack ["artist"] = audiomack["artist"].str.lower().str.strip()

In [144]:
radio["radio_count"] = pd.to_numeric(
    radio["radio_count"],
    errors="coerce"
)
youtube["youtube_count"] = pd.to_numeric(
    youtube["youtube_count"],
    errors="coerce"
)
audiomack["audiomack_count"] = pd.to_numeric(
    audiomack["audiomack_count"],
    errors="coerce"
)

# Aggregation Method

In [145]:
radio_clean = (
    radio
    .groupby ("title", as_index=False)
    ["radio_count"]
    .sum()
)

In [146]:
youtube_clean = (
    youtube
.groupby (["title", "artist"], as_index=False)
    ["youtube_count"]
    .sum()
)

In [147]:
audiomack_clean = (
    audiomack 
    .groupby(["title", "artist"], as_index=False)
    ["audiomack_count"]
    .sum()
)

# Normalization Method

In [148]:
# Platform metrics operate on different scales.
# Normalize values to a common 0-1 range so that
# radio impressions and streaming counts can be
# compared fairly.

In [149]:
#Normalize radio score
radio_clean["radio_score"] = (
    radio_clean["radio_count"] / radio_clean["radio_count"].max()
)

In [150]:
#Normalize youtube score
youtube_clean["youtube_score"] = (
    youtube_clean["youtube_count"] / youtube_clean["youtube_count"].max()
)

In [151]:
#normalize audiomark score
audiomack_clean["audiomack_score"] = (
    audiomack_clean["audiomack_count"] / audiomack_clean["audiomack_count"].max()
)



# Merging

In [152]:
combined = youtube_clean.merge(
    audiomack_clean,
    on=["title", "artist"],
    how="outer"
)

combined = combined.fillna(0)

unique_titles = (
    combined
    .groupby("title")["artist"]
    .nunique()
    .reset_index()
)

unique_titles = unique_titles[
    unique_titles["artist"] == 1
]["title"]

radio_safe = radio_clean[
    radio_clean["title"].isin(unique_titles)
]

combined = combined.merge(
    radio_safe,
    on="title",
    how="left"
)

combined = combined.fillna(0)

# Chart Score Calculation 

In [153]:
# Weighting assumption:
# YouTube = 35%
# Audiomack = 35%
# Radio = 30%
#
# The assessment did not provide official platform weights.
# These weights were chosen to reflect the importance of streaming
# in modern Nigerian music consumption while preserving radio influence

combined["Total"] = (
    combined["youtube_score"] * 0.35 +
    combined ["audiomack_score"] * 0.35 + 
    combined ["radio_score"] * 0.30
)


# Top 10 Leaderboard Output

In [154]:
top10 = (
    combined
    .sort_values(
        by="Total",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

top10["Rank"] = range(1, 11)

top10 = top10[
    [
        "Rank",
        "title",
        "artist",
        "Total"
    ]
]

top10

,Rank,title,artist,Total
0,1,higher (burna boy),burna boy,0.350000
1,2,lonely at the top,asake,0.350000
2,3,ijo (laba laba),crayon,0.305211
3,4,peace be unto you (pbuy),asake,0.303723
4,5,certified loner (no competition),mayorkun,0.295982
5,6,ogechi,boypee & hyce &,0.263350
6,7,i'm a mess,johan lenox,0.254186
7,8,overloading (overdose),"mavins, crayon and ayra starr",0.244604
8,9,benin boys,rema & shallipopi,0.223129
9,10,anabella,khaid,0.216984


# Exporting Top 10 CSV File 

In [139]:
top10.to_csv("top10_leaderboard.csv", index=False)